# UltraLight VM-UNet — train & test on RTX 5060 (Blackwell)

Self-contained v2 of the codebase for a Blackwell GPU. Reproduces Table 1 of
Wu et al., *Patterns* 6, 101298 (2025).

**Target (ISIC2017):** DSC 0.9091 · IoU 0.8334 · ACC 0.9646 · SE 0.9053 · SP 0.9790 ·
Prec 0.9481, at 0.049 M params / 0.060 GFLOPs.

### Why this directory exists

The RTX 5060 is Blackwell, compute capability **sm_120**. The `torch 2.0.1+cu117`
pinned at the repo root has kernels only up to **sm_86** and embeds just `compute_37`
PTX, so it cannot run here at all — it fails with *"no kernel image is available for
execution on the device"*. PyTorch **2.7.0+ with CUDA 12.8** was the first stable
release with native sm_120 support.

Three things differ from the root codebase; nothing that affects the result:

| | why |
|---|---|
| `timm.layers` import with fallback | timm moved `trunc_normal_` in 0.9; the old path is a shim |
| `torch.load(weights_only=False)` | torch ≥ 2.6 defaults to `True`, which **rejects** these checkpoints (`min_loss` is `np.float64`) |
| `val_batch_size = 30` | speed only — no gradients in validation. Verified: shifts val loss by 6e-5, runs 20× faster |

### Setup (run once, in a terminal, before starting this notebook)

```bash
cd blackwell
uv venv --python 3.11 .venv
.venv\Scripts\activate
uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
uv pip install -r requirements.txt
python -m ipykernel install --user --name ultralight-bw --display-name "UltraLight (Blackwell)"
```

Then pick the **UltraLight (Blackwell)** kernel for this notebook.

## 1. Preflight — can this torch actually run on this GPU?

This is the cell that matters on new hardware. It does not trust the version string;
it executes a real kernel and reports what happened.

In [ ]:
import os, sys, platform
import torch

print('python  :', platform.python_version())
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print()

assert torch.cuda.is_available(), 'no CUDA device visible to torch'
name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
archs = torch.cuda.get_arch_list()
print(f'device  : {name}')
print(f'compute : sm_{cc[0]}{cc[1]}')
print(f'VRAM    : {vram:.1f} GB')
print(f'built   : {archs}')
print()

# The definitive check: run a real kernel rather than compare version strings.
try:
    a = torch.randn(512, 512, device='cuda')
    (a @ a).sum().item()
    torch.cuda.synchronize()
    print(f'OK - torch {torch.__version__} executes on sm_{cc[0]}{cc[1]}')
except RuntimeError as e:
    raise SystemExit(
        f'torch {torch.__version__} CANNOT execute on sm_{cc[0]}{cc[1]}.\n'
        f'It was built for {archs}.\n\n'
        'Blackwell needs torch >= 2.7 with CUDA 12.8:\n'
        '  uv pip install torch torchvision '
        '--index-url https://download.pytorch.org/whl/cu128\n\n'
        f'{type(e).__name__}: {e}')

## 2. Working directory

Everything below assumes the `blackwell/` directory, so the notebook behaves the same
whether the kernel starts in `notebooks/` or elsewhere.

In [ ]:
import os, sys

here = os.getcwd()
if os.path.basename(here) == 'notebooks':
    os.chdir('..')
BW = os.getcwd()
sys.path.insert(0, BW)

assert os.path.isfile('train.py'), f'expected blackwell/ layout, got {BW}'
print('working dir:', BW)
print('contents   :', sorted(d for d in os.listdir() if not d.startswith('.'))) 

## 3. Data

Pulls the six prepared `.npy` splits (524 MB) from HuggingFace. Preprocessing and the
train/val/test split were done **once** with `SPLIT_SEED = 42`, so every machine
consumes identical bytes and the split cannot drift.

The dataset repo is private — authenticate first with `huggingface-cli login`, or set
`HF_TOKEN` in the environment.

> The split matters. An earlier run used a *sorted* file listing, which looked
> reproducible but was biased: ISIC IDs correlate with acquisition source, giving
> train/val/test mean lesion areas of 22.9% / 8.0% / 15.0%. That alone cost 4.1 DSC
> points. The seeded permutation gives 20.0% / 17.6% / 18.6%.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/hf_data.py", "pull",
                "--dataset", "ISIC2017", "--repo", "RohanRamesh/ultralight-vmunet-data"], check=True)

In [ ]:
import numpy as np

print(f'{"split":6s} {"images":>7s} {"shape":>22s} {"lesion area":>12s}')
print('-' * 52)
for s in ('train', 'val', 'test'):
    d = np.load(f'data/ISIC2017/data_{s}.npy', mmap_mode='r')
    m = np.load(f'data/ISIC2017/mask_{s}.npy', mmap_mode='r')
    fg = (np.asarray(m) >= 128).mean() * 100
    print(f'{s:6s} {len(d):7d} {str(d.shape):>22s} {fg:11.1f}%')

# balanced splits are the fix described above; wildly different values mean stale data
print()
print('expect roughly 20.0 / 17.6 / 18.6 % -- if not, the data is from the old sorted split')

## 4. Sanity checks — do not skip

Seconds to run, and they catch the failures that would otherwise surface hours in.

The Mamba layer here is a **pure-PyTorch reimplementation** (`mamba_ssm` is a CUDA
extension that does not build on Windows, and a per-machine backend switch would mean
the code you debug is not the code producing the numbers). These tests assert it still
matches the reference scan on *this* torch build — which is a newer one than the tests
were originally written against.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import torch
from models.UltraLight_VM_UNet import UltraLight_VM_UNet, MAMBA_BACKEND
from thop import profile

m = UltraLight_VM_UNet().cuda()
total = sum(p.numel() for p in m.parameters())
flops, _ = profile(m, inputs=(torch.randn(1, 3, 256, 256).cuda(),), verbose=False)

print('backend :', MAMBA_BACKEND)
print(f'params  : {total}   (paper: 49457)')
print(f'GFLOPs  : {flops/1e9:.4f}  (paper: 0.060; see README on the 0.0047 gap)')
assert total == 49457, f'parameter count drifted: {total} != 49457'
print()
print('OK - structural match to the paper')
del m; torch.cuda.empty_cache()

## 5. (optional) Where does throughput peak on this GPU

The model is 0.049 M parameters, so **VRAM is not the limiting resource** — a batch of 8
uses well under 1 GB. The cost is kernel-launch overhead: the selective scan issues ~176
sequential launches per forward *regardless of batch size*, so a larger batch amortises a
fixed cost over more images.

Useful for planning your own experiments. **Do not raise `config.batch_size` for the
replication run** — 8 is the paper's hyperparameter, and 32 would mean 39 optimiser steps
per epoch instead of 157, which changes the training trajectory and the result.

In [ ]:
!python scripts/bench_batch.py

## 6. Train

250 epochs, then automatic evaluation of the best checkpoint on the test split.

Reference cost: **1.64 h on a Kaggle T4** (~23 s/epoch). The RTX 5060 should beat that —
it is a newer architecture, and because this workload is bound by CPU launch overhead
rather than GPU throughput, a modern laptop CPU helps more than the GPU does. Validation
batching (30 instead of 1) takes roughly another 25% off.

`checkpoints/latest.pth` is written every epoch and resumed automatically, so an
interrupted run continues where it stopped.

In [ ]:
!python train.py

## 7. Test a checkpoint on its own

`train.py` already evaluates the best checkpoint at the end. This is for re-evaluating
later without retraining — after a crash, or to compare checkpoints.

In [ ]:
import glob, os

runs = sorted(glob.glob('results/UltraLight_VM_UNet_ISIC2017_*'), key=os.path.getmtime)
assert runs, 'no run directories found -- has training been run?'
latest = runs[-1]
ckpts = sorted(glob.glob(os.path.join(latest, 'checkpoints', 'best-epoch*.pth')))
print('latest run :', latest)
print('checkpoints:', [os.path.basename(c) for c in ckpts] or '(none yet)')

In [ ]:
# evaluates the best checkpoint from the run above
import glob, os, subprocess, sys

ckpts = sorted(glob.glob(os.path.join(latest, 'checkpoints', 'best-epoch*.pth')))
if ckpts:
    subprocess.run([sys.executable, 'test.py', '--weights', ckpts[-1],
                    '--work-dir', latest], check=True)
else:
    print('no best-epoch checkpoint yet; run cell 6 first')

## 8. Results vs. the paper

In [ ]:
import glob, re, os

PAPER = {'DSC': 0.9091, 'IoU': 0.8334, 'ACC': 0.9646,
         'SE': 0.9053, 'SP': 0.9790, 'Prec': 0.9481}

logs = sorted(glob.glob(os.path.join(latest, 'log', '*.log')), key=os.path.getmtime)
text = ''.join(open(p, encoding='utf-8', errors='replace').read() for p in logs)
hits = re.findall(r'test of best model.*', text)
assert hits, 'no test result in the log yet'
line = hits[-1]
print(line[:200], '\n')

def grab(k):
    m = re.search(k + r':\s*([\d.]+)', line)
    return float(m.group(1)) if m else float('nan')

tp = int(re.search(r'\[\s*(\d+)\s+(\d+)\]\]', line).group(2)) if re.search(r'\[\s*(\d+)\s+(\d+)\]\]', line) else None
ours = {'DSC': grab('f1_or_dsc'), 'IoU': grab('miou'), 'ACC': grab('accuracy'),
        'SE': grab('sensitivity'), 'SP': grab('specificity')}

print(f'{"metric":8s} {"paper":>8s} {"ours":>8s} {"delta":>9s}')
print('-' * 38)
for k in ('DSC', 'IoU', 'SE', 'SP', 'ACC'):
    d = ours[k] - PAPER[k]
    flag = '' if abs(d) <= 0.01 else '  <-- outside +/-0.01'
    print(f'{k:8s} {PAPER[k]:8.4f} {ours[k]:8.4f} {d:+9.4f}{flag}')

print()
print('The paper gives no seed for its "random" split, so a different partition of the')
print('same 2000 images lands differently. Within about +/-0.01 DSC is a successful')
print('replication; a larger gap is worth investigating.')

---

## Notes

**Comparing across machines.** This environment runs a different torch than the Kaggle T4
runs (2.11 vs 2.6-ish) and a different cuDNN, so results may differ slightly even with an
identical split and seed — convolution algorithm selection is not guaranteed stable across
versions. Fine for a standalone replication; worth stating if you put numbers from both
machines in the same table.

**8 GB is plenty.** Peak usage at batch 8 is ~0.7 GB. If you want to use the card harder,
that is what cell 5 is for — but see the warning there about `batch_size` and the paper.

**`num_workers = 0`** deliberately. Raising it hides the CPU-side `scipy.ndimage.rotate`
augmentation and would be the single biggest remaining speedup, but each worker seeds its
own RNG, so the augmentation stream — and therefore the training trajectory — changes.
Left alone so the replication stays comparable. Reasonable to enable once you are
benchmarking your own variants against your own baseline.

**Full deviation list** from the reference implementation is in `../README.md`.